# Question Answering with LangChain, OpenAI, and MultiQuery Retriever


<br>

## Intro

In this lab you'll build a Retrieval-Augmented Generation (RAG) chatbot backed by Elasticsearch and OpenAI, using LangChain's `MultiQueryRetriever` to improve retrieval quality.

**Why use a `MultiQueryRetriever`?** A single user question, phrased one way, might miss documents that use different wording for the same idea. [`MultiQueryRetriever`](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) asks an LLM to rephrase the original question into several alternative versions, runs a separate search for each version, and merges the results — giving you a broader, more relevant set of documents than a single search would.

Here's how the workflow will look like:

- **Setup (once):** split a sample dataset of documents into passages (chunks), turn each passage into an embedding, and store them in a vector database (in this case, we'll use Elasticsearch).
- **At question time:** the `MultiQueryRetriever` generates a few variations of the question, searches Elasticsearch with each one, and combines the retrieved passages. Those passages are then passed to an LLM, which uses them as context to write the final answer.


And here's what to expect from this lab:

1. **Complete a guided demo** — You'll be given the overall structure and process to follow: connecting to Elasticsearch and OpenAI, loading and chunking a sample set of workplace documents, embedding and indexing them into a vector store, and wiring up a `MultiQueryRetriever` chain that expands a question into several variations before retrieving and answering. Some steps are already implemented for you, and you'll need to complete the rest (e.g. filling in missing parameters and logic) to get it fully working.
2. **Replicate it yourself with variations** — Then, you'll create **at least two new iterations** of the question-answering step (new questions and/or settings) to explore how `MultiQueryRetriever` behaves.

By the end of this lab, you'll understand how multi-query retrieval works and be able to apply it to your own RAG pipelines.

<br>

## Install and import dependencies

Uncomment and run the cells below to install the dependencies required for this notebook.

Tip: Use a virtual environment to keep this project's dependencies isolated from your system Python and other projects.

In [5]:
%pip install "langchain>=0.3,<0.4" "langchain-core>=0.3,<0.4" "langchain-community>=0.3,<0.4" "langchain-openai>=0.3,<0.4" "langchain-elasticsearch>=0.3,<0.4" "jq==1.12.0"

  Using cached jq-1.12.0-cp313-cp313-win_amd64.whl.metadata (7.3 kB)
   ---------------------------------------- 0.0/952.8 kB ? eta -:--:--
   ---------------------------------------- 952.8/952.8 kB 11.8 MB/s  0:00:00

   ---------------- ----------------------- 2/5 [elastic-transport]
   ------------------------ --------------- 3/5 [elasticsearch]
   ------------------------ --------------- 3/5 [elasticsearch]
   ------------------------ --------------- 3/5 [elasticsearch]
   ------------------------ --------------- 3/5 [elasticsearch]
   ------------------------ --------------- 3/5 [elasticsearch]
   ------------------------ --------------- 3/5 [elasticsearch]
   ------------------------ --------------- 3/5 [elasticsearch]
   ------------------------ --------------- 3/5 [elasticsearch]
   ------------------------ --------------- 3/5 [elasticsearch]
   ------------------------ --------------- 3/5 [elasticsearch]
   ------------------------ --------------- 3/5 [elasticsearch]
   ------

In [6]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai import ChatOpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from getpass import getpass

## Connect to Elasticsearch

In this lab, you'll use Elasticsearch as the vector database and search engine that stores your documents and retrieves the most relevant ones in response to user queries.

**What is Elasticsearch?**

Elasticsearch is a distributed search and analytics engine designed for fast retrieval over large collections of data. In AI applications, it can store both text and vector embeddings, making it well suited for Retrieval-Augmented Generation (RAG) workflows where an LLM retrieves relevant context before generating a response.


**Getting started with Elastic Cloud:**

1. If you don't already have one, sign up for a free trial: [https://www.elastic.co/](https://www.elastic.co/).
2. During sign-up (or afterwards), create a deployment/project. Either a classic deployment or a serverless project will work for this lab.
3. Once your deployment/project is ready, you'll need to find your **Cloud ID** and create an **API key**. 

<br>

> To find your **Cloud ID** and create an **API key**, 
> follow the instructions on the link below.
> 
>   👇👇👇
>
> https://www.elastic.co/search-labs/tutorials/install-elasticsearch/find-cloud-id-create-api-keys 📌
>
<br>

In [10]:
#
# Serverless project: use the Elasticsearch endpoint URL from the project's home page
# (top right, "Elasticsearch: https://...") and an encoded API key ("API keys" button).
#
ELASTIC_URL = getpass("Elasticsearch endpoint URL: ")
ELASTIC_API_KEY = getpass("Elastic Api Key: ")

# https://platform.openai.com/api-keys
OPENAI_API_KEY = getpass("OpenAI API key: ")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small", openai_api_key=OPENAI_API_KEY)

vectorstore = ElasticsearchStore(
    es_url=ELASTIC_URL,
    es_api_key=ELASTIC_API_KEY,
    index_name="workplace-search-docs",
    embedding=embeddings,
)

## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [11]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [12]:
from langchain.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


def metadata_func(record: dict, metadata: dict) -> dict:
    #Populate the metadata dictionary with keys name, summary, url, category, and updated_at.
    metadata["name"] = record.get("name")
    metadata["summary"] = record.get("summary")
    metadata["url"] = record.get("url")
    metadata["category"] = record.get("category")
    metadata["updated_at"] = record.get("updated_at")  # only some records have this

    return metadata


# For more loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/
# And 3rd party loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/#third-party-loaders
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=800, chunk_overlap=400
)
docs = loader.load_and_split(text_splitter=text_splitter)

### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [14]:
documents = vectorstore.from_documents(
    docs,
    embeddings,
    index_name="workplace-search-docs",
    es_url=ELASTIC_URL,
    es_api_key=ELASTIC_API_KEY,
)

# gpt-3.5-turbo-instruct (the default of OpenAI()) was shut down on 2026-09-28, so we use GPT-6 Luna.
# temperature=1 because GPT-5/6-family reasoning models reject other values.
llm = ChatOpenAI(model="gpt-6-luna", temperature=1, openai_api_key=OPENAI_API_KEY)

retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [15]:
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document
from langchain_core.output_parsers import StrOutputParser

import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm | StrOutputParser()

ans = chain.invoke("what is the nasa sales team?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['What does the NASA sales team do?', 'How is the NASA sales team organized, and what are its responsibilities?', 'Who are the members of the NASA sales team, and what products or services do they sell?']


---- Answer ----
NASA refers to the **North America–South America sales region** within the Americas. It covers the United States, Canada, Mexico, Central America, and South America. The region is led by two Area Vice-Presidents: **Laura Martinez** for North America and **Gary Johnson** for South America. The sales teams work with customers, pursue new business, and support existing client relationships.


<br>

## 🚀 Your turn

Time to make this lab your own!


So far we've asked a single question ("what is the nasa sales team?") and looked at one answer. Now it's your turn to try the pipeline with your own questions and see how `MultiQueryRetriever` handles them.


Here's what to do:

1. **Pick 2 new questions** — Look through the sample dataset (or just get creative) and write down 2 questions you'd like the chatbot to answer, different from the example above.
2. **Ask each question** — For each question, copy the last code cell, replace the question passed to `chain.invoke(...)`, and run it.
3. **Check the logs** — Because logging is enabled, you'll see the alternative questions `MultiQueryRetriever` generated for you in the cell output. Take a look at them.
4. **Reflect** — In a markdown cell, briefly note: Did the generated questions look useful? Was the final answer correct and well grounded in the retrieved context, or did the model seem to be missing information?

<br>

💡 **Tip:**

- If an answer looks off, try asking the same question in a different way, or peek at the generated queries to see if they drifted from what you actually meant.

<br>

⭐️ **Bonus ideas:**

- Change `retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)` to instead use `vectorstore.as_retriever()` directly (no multi-query), and compare the answers you get for the same questions. Does `MultiQueryRetriever` actually help?
- Look at the `k` parameter of `as_retriever()` (e.g. `vectorstore.as_retriever(search_kwargs={"k": 3})`) to control how many passages are retrieved per query, and see how that affects the final answer.
- Edit `LLM_CONTEXT_PROMPT` to change the tone or format of the final answer (e.g. ask for a short bullet-point summary instead of a verbose answer), and see how that changes the output.
- Ask a question that isn't covered by the sample dataset at all — does the chatbot correctly say it doesn't know, or does it make something up?

<br>

## ✅ Solution

### Question 1 — wording that differs from the documents

The dataset has an "Office Pet Policy", but the question never uses the words *pet* or *policy*. This is where generating alternative phrasings should help.

In [16]:
ans = chain.invoke("Can I bring my dog to work?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['Is the workplace pet-friendly, and are employees allowed to bring dogs to the office?', 'What are the company’s policies or restrictions on bringing a dog to work?', 'Can employees bring pets, such as dogs, into the workplace?']


---- Answer ----
Yes, but you need prior approval from both your supervisor and HR before bringing your dog to the office. Submit a written request describing your dog’s breed, age, and temperament; provide proof of current vaccinations and any required licenses; and obtain written consent from everyone who shares your workspace.

Your dog must be well-behaved, house-trained, supervised, and kept on a leash or in a secure enclosure when outside your immediate work area. Dogs are not allowed in restricted areas such as meeting rooms, restrooms, and kitchen or dining areas. HR may deny or revoke approval if concerns arise.


### Question 2 — an answer spread over several documents

Work-from-home rules appear in three documents (the original policy plus an April and a May 2023 update), so a good answer has to combine them and use the latest rule.

In [17]:
ans = chain.invoke("How many days a week am I allowed to work from home?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['What is the maximum number of days per week I can work remotely?', 'How many days each week does the work-from-home policy allow me to work from home?', 'What is my weekly allowance for working from home?']


---- Answer ----
You may work from home **two days a week**. Coordinate with your supervisor and HR to set your in-office and remote workdays.


### Reflect


**Were the generated questions useful?** Yes. For both questions the `MultiQueryRetriever` produced 3 sensible rephrasings that stayed on topic, none of them drifted away from what I meant.
- For *"Can I bring my dog to work?"* the rephrasings added the words **pet**, **pet-friendly** and **policies**, which are exactly the words the "Office Pet Policy" document uses but my question didn't. This is the case multi-query retrieval is designed for.
- For the work-from-home question the rephrasings mainly swapped synonyms (*work remotely*, *weekly allowance*, *work-from-home policy*). Useful, but less needed, because the original wording already matches the documents well.

**Was the answer correct and grounded in the context?**
- **Dog question: correct and well grounded.** Every rule in the answer (approval from supervisor and HR, written request with breed/age/temperament, vaccination proof, consent from people sharing the workspace, leash or enclosure, no pets in meeting rooms, restrooms, kitchen and dining areas, HR can revoke approval) appears in the Office Pet Policy. It didn't invent anything. It left out some parts (cleaning up, liability, allergies), which is acceptable for a direct answer.
- **Work-from-home question: correct.** The answer "two days a week" matches the latest document, the May 2023 update (3 days in the office, 2 remote). This was the trickier question: there are three conflicting documents (full-time remote in March 2020, 2 days *in the office* from May 2022, then 3 days in the office from May 2023), and the model picked the most recent rule. However, it didn't say which document or date the rule came from, so a reader can't tell that older rules exist.
- **NASA demo question: mostly correct.** Region, countries and the two Area Vice-Presidents match the Sales Organization Overview. The last sentence ("work with customers, pursue new business…") is generic filler not stated about NASA in the document, a small sign of the model adding unsupported detail.

**What was missing or could be improved?**
- The prompt asks the model to be "as verbose and educational as possible", but all answers were short. The model followed the context more than that style instruction.
- With conflicting or updated documents, the answer should cite its source and date. Asking for that in `LLM_CONTEXT_PROMPT` (the `SOURCE: {name}` is already in the context), or adding `updated_at` to the document prompt, would make answers like the WFH one more trustworthy.